# Performance recording

## Simulations

In [1]:
from nanover.app import OmniRunner, RenderingSelection
from nanover.openmm import OpenMMSimulation
from controls import show_controls

simulations = {}

# 300 increments
# names = [
#     "solvent-000900.xml",
#     "solvent-001200.xml",
#     "solvent-001500.xml",
#     "solvent-001800.xml",
#     "solvent-002100.xml",
#     "solvent-002400.xml",
#     "solvent-002700.xml",
#     "solvent-003000.xml",
#     "solvent-006000.xml",
#     "solvent-009000.xml",
#     "solvent-012000.xml",
#     "solvent-015000.xml",
#     "solvent-018000.xml",
#     "solvent-021000.xml",
#     "solvent-024000.xml",
# ]

# 15k increments
# names = [
#     "solvent-015000.xml",
#     "solvent-030000.xml",
#     "solvent-045000.xml",
#     "solvent-060000.xml",
#     "solvent-075000.xml",
#     "solvent-090000.xml",
# ]

# 3k increments
names = [
    "solvent-015000.xml",
    "solvent-018000.xml",
    "solvent-021000.xml",
    "solvent-024000.xml",
    "solvent-027000.xml",
    "solvent-030000.xml",
]

for name in names:
    simulation = OpenMMSimulation.from_xml_path(name)
    simulations[name] = (simulation, ())

# preload simulations
for simulation, selections in simulations.values():
    simulation.load()

## Runner

In [ ]:
from contextlib import suppress

with suppress(NameError):
    imd_runner.close()

imd_runner = OmniRunner.with_basic_server(name="PERF-TESTS")

for name, (simulation, selections) in simulations.items():
    simulation.name = name
    imd_runner.add_simulation(simulation)
    imd_runner.set_simulation_selections(simulation, *selections)

imd_runner.load(0)
show_controls(imd_runner)

interactive(children=(Dropdown(description='Force Type', index=1, options=(('Gaussian', 'gaussian'), ('Harmoni…

interactive(children=(IntSlider(value=100, description='Force Scale', max=1000, min=1), Output()), _dom_classe…

interactive(children=(FloatSlider(value=0.4, description='Force Range', max=2.0, min=0.1, step=0.05), Output()…

interactive(children=(FloatSlider(value=1.0, description='Passthrough', max=1.0), Output()), _dom_classes=('wi…

## Recording

In [3]:
ITERATION_DURATION = 30

### Networking (hidden solvent)

In [4]:
from nanover.websocket.record import record_from_runner
from nanover.websocket import NanoverImdClient
from time import sleep

with NanoverImdClient.from_runner(imd_runner) as client:
    client.set_shared_value("suggested.report.connection.health", True)
    with client.root_selection.modify() as selection:
        selection.hide = True

record_from_runner(imd_runner, "FAKE.nanover.zip")
# record_from_runner(imd_runner, "REDO--solvent-hidden-4-user--granular.nanover.zip")

for i in range(len(simulations)):
    imd_runner.load(i)
    sleep(ITERATION_DURATION)

imd_runner.close()

### Rendering

In [ ]:
from nanover.websocket.record import record_from_runner
from nanover.websocket import NanoverImdClient
from time import sleep

with NanoverImdClient.from_runner(imd_runner) as client:
    client.set_shared_value("suggested.report.connection.health", True)

record_from_runner(imd_runner, "FAKE.nanover.zip")

for i in range(len(simulations)):
    imd_runner.load(i)
    sleep(ITERATION_DURATION)

imd_runner.close()